# Лабораторная работа №14: Управляемая генерация изображений

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Получить canny/depth/pose controls.
- Сравнить control scale и CFG.
- Использовать control как synthetic annotation для downstream model.

## Теоретическая часть

ControlNet добавляет trainable branch к замороженной diffusion model и принимает spatial condition (edges, depth, pose). В лабораторной используется готовый ControlNet; изменяются conditioning scale и CFG.

![Пример результата ControlNet с Canny-контролем](https://huggingface.co/lllyasviel/sd-controlnet-canny/resolve/main/images/bird_canny_out.png)


Готовый ControlNet добавляет пространственное условие к SD1.5. В работе контур Canny вычисляется только из train MNIST; тот же исходный label сопровождает synthetic image. Edge F1, CLIP text similarity и downstream accuracy на реальном test отражают разные свойства и могут конфликтовать.

Для контуров $F_1=2PR/(P+R)$; допустимое геометрическое отклонение задаётся расширением контуров до расчёта точности и полноты. Downstream accuracy измеряется только на реальном отложенном test.

### Экспериментальный протокол

Выбрано условие Canny, обучение ControlNet не требуется. Полный режим: 20 train контуров каждого из 10 классов, 5 генераций на контур (base + сетка scale/CFG), отдельный real test и одинаковые 500 optimizer steps каждого downstream classifier. Это дорогой эксперимент: перед запуском оцените лимит Colab, сохраняйте artifacts в Drive; FAST_DEV_RUN генерирует по 1 примеру каждого класса и 2 шага классификатора, только smoke. Нужны CUDA, diffusers transformers accelerate opencv-python torchvision.

### Материалы

- [Diffusers ControlNet](https://huggingface.co/docs/diffusers/using-diffusers/controlnet)
- [ControlNet paper](https://arxiv.org/abs/2302.05543)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.

Версии библиотек, источник данных и лицензия должны быть указаны в отчёте.

## Реализация

`FAST_DEV_RUN=True` проверяет связность пайплайна на малом бюджете и не является итоговым экспериментом. Полная сдача требует `FAST_DEV_RUN=False` и реально полученных метрик.

In [ ]:
from pathlib import Path
import os, json, random, time
import numpy as np
import torch
SEED = 42
FAST_DEV_RUN = False  # True только для короткой проверки связности, не для сдачи.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ARTIFACTS = Path(os.environ.get('CV_ARTIFACTS', 'artifacts'))
ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path(os.environ.get('CV_COURSE_DATA', str(Path.cwd().parents[1] / 'data')))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
def save_json(name, value):
    destination = ARTIFACTS / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(value, ensure_ascii=False, indent=2))
print({'device':DEVICE,'fast_dev_run':FAST_DEV_RUN,'artifacts':str(ARTIFACTS)})

### Непересекающиеся MNIST выборки

*Вес: 1 балл.*

**Постановка.** Контур и метка должны происходить из одной исходной сцены, иначе synthetic augmentation получает неверную разметку. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите только train/test MNIST, выберите реальные train индексы по классам.

**Результат.** Проверяемое состояние «Непересекающиеся MNIST выборки» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите только train/test MNIST, выберите реальные train индексы по классам» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Контуры Canny и происхождение меток

*Вес: 1 балл.*

**Постановка.** Контур и метка должны происходить из одной исходной сцены, иначе synthetic augmentation получает неверную разметку. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Постройте PNG-контуры, сохраните исходные индексы, классы и фиксированные seeds.

**Результат.** Проверяемое состояние «Контуры Canny и происхождение меток» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Постройте PNG-контуры, сохраните исходные индексы, классы и фиксированные seeds» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Загрузка предобученного ControlNet

*Вес: 1 балл.*

**Постановка.** Совместное изменение силы контроля и CFG выявляет компромисс между структурой контура и текстовым условием. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите SD1.5 и Canny ControlNet с совместимым scheduler; CPU-имитация недопустима.

**Результат.** Проверяемое состояние «Загрузка предобученного ControlNet» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите SD1.5 и Canny ControlNet с совместимым scheduler; CPU-имитация недопустима» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Факторная сетка генераций

*Вес: 1 балл.*

**Постановка.** Совместное изменение силы контроля и CFG выявляет компромисс между структурой контура и текстовым условием. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Сравните baseline и все сочетания силы контроля и CFG на одинаковых входах и seeds.

**Результат.** Проверяемое состояние «Факторная сетка генераций» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Сравните baseline и все сочетания силы контроля и CFG на одинаковых входах и seeds» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Структурное и текстовое соответствие

*Вес: 1 балл.*

**Постановка.** Не следует считать один CLIP score доказательством точной геометрии либо корректности класса цифры. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Из реальных PNG рассчитайте edge F1 с допуском и CLIP text-image cosine.

**Результат.** Проверяемое состояние «Структурное и текстовое соответствие» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Из реальных PNG рассчитайте edge F1 с допуском и CLIP text-image cosine» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Разнообразие внутри класса

*Вес: 1 балл.*

**Постановка.** Не следует считать один CLIP score доказательством точной геометрии либо корректности класса цифры. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Измерьте попарные различия изображений внутри одного класса и настройки.

**Результат.** Проверяемое состояние «Разнообразие внутри класса» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Измерьте попарные различия изображений внутри одного класса и настройки» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Аудит худших случаев

*Вес: 1 балл.*

**Постановка.** Не следует считать один CLIP score доказательством точной геометрии либо корректности класса цифры. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Сохраните построчные измерения и пять наихудших примеров, не приписывая им вымышленные оценки.

**Результат.** Проверяемое состояние «Аудит худших случаев» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Сохраните построчные измерения и пять наихудших примеров, не приписывая им вымышленные оценки» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Архитектура downstream-классификатора

*Вес: 1 балл.*

**Постановка.** Структурно похожее изображение может ухудшать downstream classification; качество проверяется на полностью реальном test. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Определите CNN и фиксированную реальную обучающую подвыборку.

**Результат.** Проверяемое состояние «Архитектура downstream-классификатора» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Определите CNN и фиксированную реальную обучающую подвыборку» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Равнобюджетное обучение с аугментацией

*Вес: 1 балл.*

**Постановка.** Структурно похожее изображение может ухудшать downstream classification; качество проверяется на полностью реальном test. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Обучите real-only и real+synthetic варианты с одинаковым числом обновлений.

**Результат.** Проверяемое состояние «Равнобюджетное обучение с аугментацией» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Обучите real-only и real+synthetic варианты с одинаковым числом обновлений» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Сравнение на реальном test

*Вес: 1 балл.*

**Постановка.** Структурно похожее изображение может ухудшать downstream classification; качество проверяется на полностью реальном test. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Сохраните точность по классам и confusion matrices на независимом real test.

**Результат.** Проверяемое состояние «Сравнение на реальном test» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Сохраните точность по классам и confusion matrices на независимом real test» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Творческое мини-исследование

*Вес: 1 балл, бонус сверх десяти обязательных заданий.*

**Постановка.** Контролируемая абляция позволяет отделить влияние одного фактора от шума выборки. Гипотеза формулируется до просмотра отложенных результатов.

Выполните следующие действия:

1. Сформулируйте собственную предметную гипотезу о поведении модели и измените один фактор, сохраняя те же реальные входы, seed и вычислительный бюджет.
2. Сравните с основным экспериментом, не используя test для выбора настройки; объясните наблюдаемые расхождения.

**Результат.** Отдельные код, измеренная таблица и интерпретация в отчёте.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена на реальных входах, сохранены фактические измерения и показана контролируемая разница. Бонус не является обязательным для Run All.

## Анализ результатов

Сопоставьте конфигурации при одинаковых данных, seeds и бюджете. Приведите таблицу фактических метрик и времени, не менее пяти неудачных примеров с объяснением причин, ограничения и проверяемые выводы. Не подменяйте измерения утверждением об успешном запуске.

## Требования к сдаче

Перед сдачей перезапустите ядро и выполните все ячейки по порядку в полном режиме. Приложите конфигурацию, версии библиотек, источники/лицензии данных и модели, checkpoints, таблицу измерений, изображения и краткий анализ ошибок. Для длительного обучения перенесите каталог `artifacts/` на Google Drive и возобновляйте обучение с последнего checkpoint; отсутствие GPU или данных нельзя заменять синтетическими метриками.